In [ ]:
# Imports
import re, unicodedata
import numpy as np
import pandas as pd
from pathlib import Path

# File paths
NBA_TGT_PATH = '/content/drive/MyDrive/nba-draft-sucess/final_master/nba_stats_with_labels.csv'
COLLEGE_PATH = '/content/drive/MyDrive/nba-draft-sucess/stat_scraper/college_players_stats.csv'
COMBINE_PATH = '/content/drive/MyDrive/nba-draft-sucess/stat_scraper/nba_combine.csv'

OUT_DIR      = '/content/drive/MyDrive/nba-draft-sucess/final_master'
FEAT_MATCHED = 'college_combine_features_matched.csv'
FEAT_NEW     = 'college_combine_new_players.csv'
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)


In [ ]:
# Methods for name cleaning and numeric conversion

# Suffixes like Jr., Sr., II, III, etc. we want to ignore
SUFFIXES = {"jr","jr.","sr","sr.","ii","iii","iv","v"}

def strip_accents(s: str) -> str:
    """Remove accents/diacritics from a string."""
    return ''.join(ch for ch in unicodedata.normalize('NFKD', s) if not unicodedata.combining(ch))

def clean_tokens(tokens):
    """Lowercase, remove punctuation, and drop suffixes."""
    out = []
    for t in tokens:
        t = strip_accents(t.strip().lower())
        t = re.sub(r"[^\w\s-]", "", t)
        if t and t not in SUFFIXES:
            out.append(t)
    return out

def to_num(s):
    """Convert to numeric; return NaN on errors."""
    return pd.to_numeric(s, errors="coerce")

def firstlast_from_freeform(name: str) -> str:
    """'First [Middle] Last' -> 'first last' (keeps multi-part last names)."""
    if not isinstance(name, str) or not name.strip():
        return ""
    toks = clean_tokens(name.split())
    if len(toks) >= 2:
        first = toks[0]
        last  = " ".join(toks[1:])
        key = f"{first} {last}"
    else:
        key = " ".join(toks)
    return re.sub(r"\s+"," ",key).strip()

def firstlast_from_lastcommafirst(name: str) -> str:
    """'Last, First [Middle]' -> 'first last'."""
    if not isinstance(name, str) or not name.strip():
        return ""
    s = name.strip()
    if "," in s:
        last, firsts = s.split(",", 1)
        last_tokens  = clean_tokens(last.split())
        first_tokens = clean_tokens(firsts.split())
        if not first_tokens or not last_tokens:
            return ""
        key = f"{first_tokens[0]} {' '.join(last_tokens)}"
    else:
        key = firstlast_from_freeform(name)
    return re.sub(r"\s+"," ",key).strip()


In [ ]:
#  Load the NBA targets file

nba = pd.read_csv(NBA_TGT_PATH, dtype=str, keep_default_na=False)

# Quick Check
if "Player" not in nba.columns:
    raise KeyError("Expected 'Player' column in nba_stats_with_labels.csv")

# Build a name join_key like 'first last'
nba["join_key"] = nba["Player"].apply(firstlast_from_freeform)
nba_keys = set(nba["join_key"])

print("NBA target list — unique players:", len(nba_keys))


In [ ]:
# Load college and combine raw files as strings

college = pd.read_csv(COLLEGE_PATH, dtype=str, keep_default_na=False)
combine = pd.read_csv(COMBINE_PATH, dtype=str, keep_default_na=False)

# Strip whitespace from string columns
for df_tmp in (college, combine):
    for c in df_tmp.columns:
        if df_tmp[c].dtype == object:
            df_tmp[c] = df_tmp[c].str.strip()

print("College rows (raw):", len(college), "| Combine rows (raw):", len(combine))


In [ ]:
#  Remove rows that include "Did not play" anywhere in the row

mask_played = ~college.apply(
    lambda r: r.astype(str).str.contains("Did not play", case=False, na=False)
).any(axis=1)
college = college[mask_played].copy()

# Drop rows with zero games

g_col = None
for cand in ["G_per_game","G"]:
    if cand in college.columns:
        g_col = cand
        break

if g_col:
    college[g_col] = to_num(college[g_col])
    college = college[college[g_col] > 0]

# Identify player name column and build join_key

player_college_col = next((c for c in college.columns if re.search(r'player|name', c, re.I)), None)
if not player_college_col:
    raise KeyError("Could not find a player-name column in the college CSV.")

college["join_key"] = college[player_college_col].apply(firstlast_from_freeform)


In [ ]:
# Choose which college stats to average (only keep those that exist)

college_stats = [
    "G_per_game","GS_per_game","MP_per_game","FG","FGA","FG%","3P","3PA","3P%","2P","2PA","2P%",
    "eFG%","FT","FTA","FT%","ORB","DRB","TRB","AST","STL","BLK","TOV","PF","PTS",
    "PER","TS%","3PAr","FTr","PProd","ORB%","DRB%","TRB%","AST%","STL%","BLK%","TOV%","USG%",
    "OWS","DWS","WS","WS/40","OBPM","DBPM","BPM"
]
college_stats = [c for c in college_stats if c in college.columns]

# Convert to numeric where applicable
for c in college_stats:
    college[c] = to_num(college[c])

# Group by join_key and average
college_avg = college.groupby("join_key")[college_stats].mean(numeric_only=True).reset_index()

print("College players after cleaning/averaging:", len(college_avg))


In [ ]:
#  Make sure the combine file has the 'PLAYER' column like 'Last, First'

if "PLAYER" not in combine.columns:
    raise KeyError("Combine CSV must include 'PLAYER' as 'Last, First'.")

# Build join_key in 'first last' form
combine["join_key"] = combine["PLAYER"].apply(firstlast_from_lastcommafirst)

# Keep useful columns if present
combine_keep = [
    "POS","HGT","WGT","BMI","BF","WNGSPN","STNDRCH","HANDL","HANDW",
    "STNDVERT","LPVERT","LANE","SHUTTLE","SPRINT","BENCH","BAR","PAN","PBHGT","PDHGT"
]
combine_keep = ["join_key"] + [c for c in combine_keep if c in combine.columns]

# Drop duplicate rows per player
combine_slim = combine[combine_keep].drop_duplicates(subset=["join_key"])

print("Combine unique players:", len(combine_slim))


In [ ]:
# Inner join ensures we only keep players in BOTH college and combine

features_all = pd.merge(college_avg, combine_slim, on="join_key", how="inner")

print("College∩Combine rows:", len(features_all))


In [ ]:
# Split based on whether the player appears in the NBA target list

features_matched = features_all[features_all["join_key"].isin(nba_keys)].copy()
features_new     = features_all[~features_all["join_key"].isin(nba_keys)].copy()

print("Matched (for training):", len(features_matched))
print("New / Unseen (for inference):", len(features_new))


In [ ]:
# Round all numeric columns to 2 decimals

def round_numeric(df):
    for c in df.columns:
        if c == "join_key":
            continue
        df[c] = to_num(df[c])
        df[c] = df[c].round(2)
    return df

features_matched = round_numeric(features_matched)
features_new     = round_numeric(features_new)


In [ ]:
# Save both outputs and print quick info

path_matched = f"{OUT_DIR}/{FEAT_MATCHED}"
path_new     = f"{OUT_DIR}/{FEAT_NEW}"

features_matched.to_csv(path_matched, index=False)
features_new.to_csv(path_new, index=False)

print(f"\n Saved TRAINING FEATURES (aligned to NBA file): {path_matched}  | rows: {len(features_matched)}")
print(f" Saved NEW/UNSEEN FEATURES (for inference):   {path_new}      | rows: {len(features_new)}")
